# Diniz Ch.5 — RLS system identification, real and complex

比較 RLS/LMS 的 spread 敏感度，並在同一 4-QAM 資料上驗證 RLS 與 RLSAlt 的 MSE、MSD 及係數歷程。

來源：`Diniz-Chap5-SysID.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Real plant and spread cases

In [ ]:
from adsp.experiments import diniz

n_samples, n_trials, seed, noise_var = 1000, 200, 501, 1e-4
results = {}
for spread in (1, 20, 80):
  make_data = partial(data.system_identification, n_samples, h=diniz.SYSTEM_TAPS, pole=diniz.pole_for_spread(spread),
                      noise_var=noise_var, burn_in=512)
  models = {
    f"RLS χ={spread}": partial(af.RLS, filter_order=7, lam=0.99, delta=0.01),
    f"LMS χ={spread}": partial(af.LMS, filter_order=7, mu=0.01)
  }
  results.update(training.compare(models, make_data, n_trials=n_trials, seed=seed))

## 3. Real-system learning curves

In [ ]:
plot.learning(results, noise_floor=noise_var)
plot.show()
training.summary(results, noise_floor=noise_var)

## 4. Complex plant and equivalent RLS forms

In [ ]:
make_qam = partial(data.system_identification, 500, h=diniz.QAM_RLS_WEIGHTS, coefficient_convention="stored",
                   input_kind="qam4", input_variance=1.0, noise_var=0.3)
initial_inverse_power = (1 - 0.99) / 1.0
models = {
  cls.__name__: partial(cls, filter_order=3, lam=0.99, delta=initial_inverse_power)
  for cls in (af.RLS, af.RLSAlt)
}
qam_results = training.compare(models, make_qam, n_trials=20, seed=seed)
fig, grid = plot.axes(rows=2, cols=2)
plot.learning(qam_results, ax=grid[0, 0], noise_floor=0.3)
plot.learning(qam_results, quantity="msd", ax=grid[0, 1], title="Complex coefficient MSD")
for ax, component in zip(grid[1], ("real", "imag")):
  plot.coefficients(qam_results["RLS"], reference=diniz.QAM_RLS_WEIGHTS, component=component, ax=ax)
plot.show(fig)
difference = np.max(np.abs(qam_results["RLS"].w_mean - qam_results["RLSAlt"].w_mean))
print("Maximum RLS/RLSAlt coefficient difference:", difference)
assert difference < 1e-8
training.summary(qam_results, noise_floor=0.3)